In [1]:
"""
data_loader.py

Downloads viral genomic sequences from NCBI using the
official NCBI Datasets CLI (ncbi-datasets-cli), querying
by NCBI Taxonomy ID.

Taxonomy IDs match the 30 virus types in Table 1 of:
Khodaei et al., BMC Bioinformatics (2025) 26:159
"""


'\ndata_loader.py\n\nDownloads viral genomic sequences from NCBI using the\nofficial NCBI Datasets CLI (ncbi-datasets-cli), querying\nby NCBI Taxonomy ID.\n\nTaxonomy IDs match the 30 virus types in Table 1 of:\nKhodaei et al., BMC Bioinformatics (2025) 26:159\n'

In [2]:

import os
import glob
import subprocess
import zipfile
import shutil
import time
import pandas as pd
from pathlib import Path
from Bio import SeqIO, Entrez
from tqdm import tqdm


In [12]:


# ── Paths ────────────────────────────────────────────────────────
DATA_DIR      = Path("../data")
RAW_DIR       = DATA_DIR / "raw"
FASTA_DIR     = DATA_DIR / "fasta"
PROCESSED_DIR = DATA_DIR / "processed"
TRIAL         = DATA_DIR / "trial"

DATA_VIRUSES_DIR = Path("C:\\Users\\rodri\\Desktop\\MASTER\\EMMBIOME\\Grecia\\Courses\\Bioinformatics\\Project\\genomic_multimodal_fusion\\Data_Viruses\\Data_Viruses.xlsx")

for d in [RAW_DIR, FASTA_DIR, PROCESSED_DIR, TRIAL]:
    d.mkdir(parents=True, exist_ok=True)


In [23]:
# ── Taxonomy ID map ──────────────────────────────────────────────
# NCBI Taxonomy IDs for the 30 viruses in Table 1 of the paper.
# You can verify any ID at: https://www.ncbi.nlm.nih.gov/taxonomy
VIRUS_TAXONOMY = {
    # Virus name        : (taxonomy_id, label_int)
    "corona"       : (694009,   0),   # Betacoronavirus
    "flu"         : (197911,   1),   # Alphainfluenzavirus
    "norwalk"           : (142786,   2),   # Norwalk virus
    "coxsackievirus"    : (12059,    3),
    "crimean-congo"     : (3052518,  4),   # CCHF virus
    "cytomegalovirus"   : (10358,    5),   # Human cytomegalovirus
    "ebola"             : (186538,   6),   # Zaire ebolavirus
    "encephalitis"      : (11072,    7),   # Eastern equine encephalitis
    "hepatitis b"       : (10407,    8),
    "hepatitis c"       : (11103,    9),
    "varicella"         : (10335,    10),  # Varicella-zoster virus
    "hiv"               : (11676,    11),  # HIV-1
    "lassa"             : (11620,    12),
    "marburg"           : (11269,    13),
    "mastadenovirus"    : (10509,    14),
    "measles"           : (11234,    15),
    "metapneumovirus"   : (162145,   16),  # Human metapneumovirus
    "mumps"             : (11161,    17),
    "papilloma"         : (10566,    18),  # Human papillomavirus
    "parainfluenza"     : (11552,    19),
    "poliovirus"        : (12110,    20),
    "poxmonkey"        : (10244,    21),  # Monkeypox virus
    "respirovirus"      : (11552,    22),  # Sendai virus family
    "rhinovirus"        : (12110,    23),  # Enterovirus (rhinovirus)
    "rotavirus"         : (10912,    24),
    "rsv"               : (11250,    25),  # Respiratory syncytial virus
    "rubella"           : (11041,    26),
    "zika"              : (64320,    27),
    "yellow fever"      : (11089,    28),
    "dengue"            : (12637,    29),
}


In [5]:
# Instead of downloading virus sequences with taxonomy IDs, we will access the Data_Viruses.xlsx
# file and try to download the sequences using the accession numbers. Each page of the document 
# corresponds to a virus, and the first column contains the accesion numbers.

print("Excel file path:", DATA_VIRUSES_DIR)
print("File exists:", DATA_VIRUSES_DIR.exists())

# Load Excel file and check available sheets
xls = pd.ExcelFile(DATA_VIRUSES_DIR)
print("Available virus sheets:", xls.sheet_names[:5], "...")  # Show first 5


Excel file path: C:\Users\rodri\Desktop\MASTER\EMMBIOME\Grecia\Courses\Bioinformatics\Project\genomic_multimodal_fusion_2\Data_Viruses\Data_Viruses.xlsx
File exists: True
Available virus sheets: ['Corona - 65060', 'Flu - 53176', 'Norwalk - 2687', 'Coxsackievirus - 1410', 'Crimean-Congo - 145'] ...


In [10]:
# Function to clean accession numbers (remove quotes)
def clean_accession(acc):
    """Remove quotes and extra characters from accession numbers"""
    if isinstance(acc, str):
        return acc.strip("'\"")  # Remove both single and double quotes
    return str(acc)

print(xls.sheet_names[:])

virus_user = str(input("Enter the virus name from the list: ")).lower().strip()

# Read input virus data as example
virus_user_sheet = [s for s in xls.sheet_names if virus_user in s.lower()]
if virus_user_sheet:
    virus_user_df = pd.read_excel(DATA_VIRUSES_DIR, sheet_name=virus_user_sheet[0])
    print(f"\nLoaded {virus_user_sheet[0]} sheet")
    print("Shape:", virus_user_df.shape)
    print("Column name:", virus_user_df.columns[0])
    
    # Clean the accession numbers
    virus_user_df['accession_clean'] = virus_user_df.iloc[:, 0].apply(clean_accession)
    print("\nFirst 5 cleaned accessions:")
    print(virus_user_df['accession_clean'].head())
    
    # Show the data
    virus_user_df.head()
else:
    print("No  sheet found for that virus!")

['Corona - 65060', 'Flu - 53176', 'Norwalk - 2687', 'Coxsackievirus - 1410', 'Crimean-Congo - 145', 'Cytomegalovirus - 312', 'Ebola - 2186', 'Encephalitis - 133', 'Hepatitis B - 8388', 'Hepatitis C - 794', 'Varicella - 72', 'HIV - 18357', 'Lassa - 263', 'Marburg - 58', 'Mastadenovirus - 1392', 'Measles - 575', 'Metapneumovirus - 356', 'Mumps - 550', 'Papilloma - 1136', 'Parainfluenza - 34 ', 'poliovirus - 723', 'PoxMonkey - 612', 'Respirovirus - 468', 'Rhinovirus -1559', 'Rotavirus - 1966', 'RSV - 1836', 'Rubella - 107 ', 'Zika - 603', 'Yellow fever - 207', 'Dengue - 8063']

Loaded Marburg - 58 sheet
Shape: (57, 1)
Column name: 'KY047763.1'

First 5 cleaned accessions:
0    NC_024781.1
1     KU179482.1
2     KC545387.1
3     KC545388.1
4     JX458827.1
Name: accession_clean, dtype: object


In [12]:
# Now it's time to download the sequences with the cleaned accession numbers.
# For that purpose, we will use the Biopython Entrez module to fetch the sequences from NCBI

# Set Entrez email
Entrez.email = "rodriguezgbalberto@gmail.com"

# Create a folder for the virus if it doesn't exist
(FASTA_DIR / virus_user).mkdir(parents=True, exist_ok=True)

for accession in tqdm(virus_user_df['accession_clean'], desc="Downloading sequences"):
    try:
        # Fetch the sequence in FASTA format
        handle = Entrez.efetch(db="nucleotide", id=accession, rettype="fasta", retmode="text")
        record = SeqIO.read(handle, "fasta")
        handle.close()
        
        # Save the sequence to a FASTA file in the specific virus folder
        fasta_path = FASTA_DIR / virus_user / f"{accession}.fasta"
        SeqIO.write(record, fasta_path, "fasta")
    except Exception as e:
        print(f"Error fetching {accession}: {e}")

Error fetching KC545387.1: Remote end closed connection without response


Error fetching JX458844.1: Remote end closed connection without response


Error fetching EU500827.1: Remote end closed connection without response


Error fetching DQ447649.1: Remote end closed connection without response


In [24]:
# This code moves all the singular fasta files into a single csv file with 5 columns:
# accession, sequence, length, virus, label
# All the viruses in the fasta folder will be processed and added to the same csv file,
# which will be saved in the processed folder as "all_viruses_sequences.csv"

def process_fasta_to_csv():
    """
    Process all FASTA files from virus subfolders and combine into a single CSV.
    Creates a DataFrame with columns: accession, sequence, length, virus, label
    """
    all_records = []
    
    # Get all virus subfolders in FASTA_DIR
    virus_folders = [f for f in FASTA_DIR.iterdir() if f.is_dir()]
    
    print(f"Found {len(virus_folders)} virus folders: {[f.name for f in virus_folders]}")
    
    for virus_folder in tqdm(virus_folders, desc="Processing virus folders"):
        virus_name = virus_folder.name

        # Get the label from VIRUS_TAXONOMY if available
        label = VIRUS_TAXONOMY.get(virus_name, -1)  # Default to -1 if not found
        if label != -1:
            label = label[1]    # Labels are stored from the Taxonomy list as (tax_id, label)
        
        # Find all FASTA files in this virus folder
        fasta_files = list(virus_folder.glob("*.fasta")) + list(virus_folder.glob("*.fa"))
        
        print(f"Processing {virus_name}: {len(fasta_files)} FASTA files")
        
        for fasta_file in tqdm(fasta_files, desc=f"Reading {virus_name} sequences", leave=False):
            try:
                # Parse the FASTA file
                for record in SeqIO.parse(fasta_file, "fasta"):
                    # Clean the sequence (keep only ACGT)
                    seq_str = str(record.seq).upper()
                    clean_seq = "".join(c for c in seq_str if c in "ACGT")
                    
                    # Skip very short sequences
                    if len(clean_seq) < 100:
                        continue
                    
                    # Create record
                    record_data = {
                        "accession": record.id,
                        "sequence": clean_seq,
                        "length": len(clean_seq),
                        "virus": virus_name,
                        "label": label
                    }
                    
                    all_records.append(record_data)
                    
            except Exception as e:
                print(f"Error processing {fasta_file}: {e}")
                continue
    
    # Create DataFrame
    df = pd.DataFrame(all_records)
    
    # Save to CSV
    output_path = PROCESSED_DIR / "all_viruses_sequences.csv"
    df.to_csv(output_path, index=False)
    
    print("✓ Processing complete!")
    print(f"Total sequences: {len(df)}")
    print(f"Saved to: {output_path}")
    
    # Show summary by virus
    if len(df) > 0:
        print("\n── Dataset Summary ─────────────────────────────")
        summary = df.groupby("virus").agg(
            count=("accession", "count"),
            mean_length=("length", "mean"),
            min_length=("length", "min"),
            max_length=("length", "max"),
        ).round(0)
        print(summary.to_string())
    
    return df

# Run the processing
df_all = process_fasta_to_csv()

# Show sample of the final dataset
print("\n── Sample of Final Dataset ─────────────────────")
print(df_all.head(10))

Found 2 virus folders: ['crimean-congo', 'marburg']


Processing virus folders:   0%|          | 0/2 [00:00<?, ?it/s]

Processing crimean-congo: 144 FASTA files


Processing virus folders:  50%|█████     | 1/2 [00:00<00:00,  2.61it/s]

Processing marburg: 56 FASTA files


Processing virus folders: 100%|██████████| 2/2 [00:00<00:00,  3.25it/s]


✓ Processing complete!
Total sequences: 200
Saved to: ..\data_genomic_multimodal_fusion_2\processed\all_viruses_sequences.csv

── Dataset Summary ─────────────────────────────
               count  mean_length  min_length  max_length
virus                                                    
crimean-congo    144      12105.0       11838       12182
marburg           56      19109.0       18997       19114

── Sample of Final Dataset ─────────────────────
    accession                                           sequence  length  \
0  DQ211617.1  TCTCAAAGATATCAATCCCCCCCTTTACCCACGTTGACACAGTCAG...   12151   
1  DQ211618.1  TCTCAAAGATATCAATCCCCCCTTTACCCACGTTGACACAGTCAGC...   12149   
2  DQ211619.1  TCTCAAAGATATCAATCCCCCCTTTACCCACATTGTCGCAGACGGC...   12176   
3  DQ211620.1  TCTCAAAGATATCAATCCCCCCATTACCCACGTTTACACAGAGAGC...   12157   
4  DQ211621.1  TCTCAAAGATATCAATCCCCCCGTTACCCACCTTGACACAGAGAGC...   12158   
5  DQ211622.1  TCTCAAAGATATCAATCCCCCCGTTACCCACGTTTACACAGAGAGC...   12157   
6  DQ21162

In [25]:
print(df_all["label"].value_counts())

label
4     144
13     56
Name: count, dtype: int64


In [1]:
import pandas as pd

df = pd.read_csv("C:\\Users\\rodri\\Desktop\\MASTER\\EMMBIOME\\Grecia\\Courses\\Bioinformatics\\Project\\genomic_multimodal_fusion\\data\\processed\\all_viruses_sequences.csv")

print(df["label"].value_counts())

label
-1    173182
Name: count, dtype: int64
